### ECC
一种非对称加密算法

基于椭圆曲线上的离散对数问题 (ECDLP)

提出者: Neal Koblitz 和 Victor S. Miller (1985)

#### 算法过程

1. 选取有限域 $\mathbb{F}_p$ 上的椭圆曲线参数
   
   曲线方程: $y^2 = x^3 + a x + b \pmod p$,满足 $4a^3 + 27b^2 \not\equiv 0 \pmod p$(非奇异)

In [1]:
# 选取一个小的素数域和曲线(便于演示)
p = 97
a = 2
b = 3
print(f"曲线: y^2 = x^3 + {a}x + {b} (mod {p})")

曲线: y^2 = x^3 + 2x + 3 (mod 97)


2. 寻找一个基点 $G$(阶为 $n$)

   $G$ 是曲线上一个点,其阶 $n$ 是一个大素数(实际中)

In [2]:
# 定义椭圆曲线上的点运算
def inv_mod(x, p):
    return pow(x, -1, p)  # Python 3.8+ 支持

class Point:
    def __init__(self, x, y, a, b, p):
        self.x = x
        self.y = y
        self.a = a
        self.b = b
        self.p = p
        # 无穷远点用 (None, None) 表示
        if x is None and y is None:
            self.inf = True
        else:
            self.inf = False

    def __eq__(self, other):
        return (self.x == other.x and self.y == other.y and 
                self.a == other.a and self.b == other.b and self.p == other.p)

    def __neg__(self):
        return Point(self.x, (-self.y) % self.p, self.a, self.b, self.p)

    def __add__(self, other):
        if self.inf:
            return other
        if other.inf:
            return self
        if self == -other:  # 互为相反数,和为无穷远点
            return Point(None, None, self.a, self.b, self.p)
        if self == other:  # 倍点
            s = (3 * self.x * self.x + self.a) * inv_mod(2 * self.y, self.p) % self.p
        else:
            s = (other.y - self.y) * inv_mod(other.x - self.x, self.p) % self.p
        x3 = (s * s - self.x - other.x) % self.p
        y3 = (s * (self.x - x3) - self.y) % self.p
        return Point(x3, y3, self.a, self.b, self.p)

    def __rmul__(self, k):
        # 标量乘法(二进制展开)
        result = Point(None, None, self.a, self.b, self.p)  # 无穷远点
        base = self
        while k > 0:
            if k & 1:
                result = result + base
            base = base + base
            k >>= 1
        return result

    def __str__(self):
        if self.inf:
            return "O"
        return f"({self.x}, {self.y})"

# 寻找一个基点：尝试 x=0..p-1,找到第一个使得 y^2 为二次剩余的点
G = None
for x in range(p):
    rhs = (x**3 + a*x + b) % p
    # 检查是否为二次剩余(即存在 y)
    # 通过尝试 y 找平方根(p 较小,暴力)
    for y in range(p):
        if (y*y) % p == rhs:
            G = Point(x, y, a, b, p)
            break
    if G is not None:
        break

print(f"G = {G}")

# 计算 G 的阶 n(暴力,实际中应使用 Schoof 算法,这里仅演示)
n = 1
P = G
while not P.inf:
    P = P + G
    n += 1
print(f"n = {n}")

G = (0, 10)
n = 50


3. 私钥 $d$(随机整数,$1 < d < n$)

In [3]:
import random
d = random.randint(2, n-1)
print(f"d = {d}")

d = 39


4. 公钥 $Q$：$Q = d \cdot G$

In [4]:
Q = d * G
print(f"Q = {Q}")

Q = (17, 10)


5. 加密(ElGamal 风格)

   将消息映射到椭圆曲线上的一个点 $M$.

   选择随机整数 $k$($1 < k < n$),计算：

   $$C_1 = k \cdot G$$
   $$C_2 = M + k \cdot Q$$

   密文为 $(C_1, C_2)$

In [5]:
# 假设消息就是基点 G(实际中需将消息编码为点)
M = G  # 仅为演示
print(f"消息点 M = {M}   (即基点本身,仅为演示)")

k = random.randint(2, n-1)
print(f"随机数 k = {k}")

C1 = k * G
C2 = M + (k * Q)
print(f"C1 = {C1}")
print(f"C2 = {C2}")

消息点 M = (0, 10)   (即基点本身,仅为演示)
随机数 k = 16
C1 = (92, 81)
C2 = (30, 0)


6. 解密：

   $$M = C_2 - d \cdot C_1$$

In [6]:
M_dec = C2 + (- (d * C1))
print(f"解密得到的消息点 M' = {M_dec}")
print(f"与原消息一致吗? {M_dec == M}")

解密得到的消息点 M' = (0, 10)
与原消息一致吗? True


#### 安全性
传播: 曲线参数 $(p, a, b, G, n, Q)$,密文 $(C_1, C_2)$

解密需要私钥 $d$

$$d \cdot G = Q \quad \text{(椭圆曲线离散对数问题)}$$

ECDLP 是目前公认的困难问题,其安全性远高于同等密钥长度的 RSA.